In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.11 Nonlinear Dynamics and Chaos: When Integrability Fails

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume II — Classical (Analytical) Mechanics",
    number="2.11",
    title="Nonlinear Dynamics and Chaos: When Integrability Fails",
    blurb="The other half of mechanics: what happens when no clever "
    "coordinates exist. Period-doubling and the Feigenbaum constant that "
    "belongs to no equation in particular, the kicked rotor's tori breaking "
    "one by one, and the Lorenz attractor measured with the discipline this "
    "volume has been building all along.",
    difficulty="advanced",
    estimate="130–160 min",
)

## Notebook overview

[§2.10](hamilton-jacobi.ipynb) closed the integrable half of mechanics: when
a full set of action variables exists, motion winds forever on invariant
tori, and the Outlook admitted the catch — *when no separation exists the
tori break*. This notebook is that horizon, made quantitative. The course
has already met chaos three times informally: the driven pendulum's
period-doubling cascade in [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb),
the double pendulum's positive Lyapunov exponent in
[§1.3](../01-elementary-mechanics/double-pendulum.ipynb), and the
Hénon–Heiles sections of [§5.5](../05-classical-stat-mech/ergodicity.ipynb)
(a Volume V notebook that borrows this one's subject for statistical ends).
What none of them did is *measure the universal numbers* — and universal
numbers, it turns out, is exactly what chaos has.

We work in three acts. First the logistic map, the fruit fly of dynamics
{cite}`may1976`: one quadratic formula whose period-doubling cascade
converges at the rate $\delta = 4.669\,201\ldots$, a constant we measure to
five digits and then measure *again* in a completely different map, because
Feigenbaum's discovery {cite}`feigenbaum1978` is that the constant belongs
to the *transition*, not the equation. Second the kicked rotor, whose
stroboscopic standard map {cite}`chirikov1979` shows Hamiltonian chaos the
way Volume II should see it: resonance islands widening as $\sqrt{K}$, the
last invariant torus dying at Greene's $K_c = 0.9716$ {cite}`greene1979`,
and beyond it deterministic motion that diffuses like a random walk, with a
diffusion coefficient we predict. Third the Lorenz system
{cite}`lorenz1963`, where we measure the largest Lyapunov exponent of a
flow two independent ways and reproduce Lorenz's own beautiful argument —
a one-dimensional return map hiding inside a three-dimensional butterfly —
that the attractor can contain no stable orbit at all. The standard text
for everything here is Strogatz {cite}`strogatz`.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the answer
is wrong; it means the output did not match what the check expected, which
may be a genuine error, a different-but-valid convention, or too tight a
tolerance. Treat a ✗ as a prompt to locate the discrepancy. Passing is
strong evidence, not proof.

## Theory in brief

**Maps as dynamics.** A first-order recursion $x_{n+1} = f(x_n)$ is the
simplest dynamical system there is, and everything this notebook needs
already lives in it. Our laboratory specimen is the logistic map

```{math}
:label: eq-nl-logistic
x_{n+1} \;=\; r\,x_n\,(1 - x_n), \qquad x \in [0, 1],\; r \in [0, 4],
```

May's toy model of a population with reproduction rate $r$ and crowding
{cite}`may1976`. A *fixed point* $x^* = f(x^*)$ attracts when
$|f'(x^*)| < 1$ and repels when $|f'(x^*)| > 1$: perturb by $\epsilon$ and
one step gives $f(x^* + \epsilon) \approx x^* + f'(x^*)\,\epsilon$, so the
derivative is the per-step amplification factor. For {eq}`eq-nl-logistic`
the non-trivial fixed point is $x^* = 1 - 1/r$ with $f'(x^*) = 2 - r$:
stable for $1 < r < 3$, and at exactly $r = 3$ the derivative reaches $-1$
and the fixed point hands over to a period-2 cycle (a *period-doubling
bifurcation*). The cycle then doubles again at $r = 1 + \sqrt 6 =
3.449\,49\ldots$, and again, and again, the doublings accumulating at
$r_\infty = 3.569\,95\ldots$ beyond which chaos begins.

**Feigenbaum universality.** The bifurcation parameters converge
geometrically, and their rate is the discovery. Using the *superstable*
parameters $R_n$ (those where the $2^n$-cycle passes through the map's
maximum $x_c = 1/2$, so the cycle derivative is exactly zero: the
numerically cleanest markers of each doubling), the ratios

```{math}
:label: eq-nl-delta
\delta_n \;=\; \frac{R_n - R_{n-1}}{R_{n+1} - R_n}
\;\longrightarrow\; \delta \;=\; 4.669\,201\,609\ldots
```

converge to Feigenbaum's constant, and the cycle's distance scale shrinks
by $\alpha = -2.502\,907\ldots$ per doubling. The stunning part, proved by
renormalization: $\delta$ and $\alpha$ are the *same for every map with a
quadratic maximum* — the logistic map, $r\sin(\pi x)$, the driven pendulum
of [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb), a
dripping faucet, convecting helium. We verify universality the honest way,
by measuring $\delta$ twice in two unrelated maps.

**The Lyapunov exponent of a map.** Chaos is exponential sensitivity, and
for a map the growth rate of an infinitesimal error after $n$ steps is the
product of the per-step derivatives, so its logarithmic average

```{math}
:label: eq-nl-lyap
\lambda \;=\; \lim_{n\to\infty} \frac{1}{n} \sum_{k=0}^{n-1}
\ln\bigl|f'(x_k)\bigr|
```

is the map's Lyapunov exponent: $\lambda < 0$ on periodic attractors,
$\lambda = 0$ at each bifurcation (the handover moment where $|f'| = 1$
around the cycle), $\lambda > 0$ in chaos. One value is known *exactly*:
at $r = 4$ the substitution $x = \sin^2(\pi y / 2)$ conjugates
{eq}`eq-nl-logistic` to the tent map, each step of which doubles $y$-errors,
so $\lambda(4) = \ln 2$ — an analytic benchmark for the numerical average.

**Hamiltonian chaos: the kicked rotor.** Volume II's own route to chaos
starts from [§2.10](hamilton-jacobi.ipynb)'s action-angle picture. Take the
freest system imaginable — a rigid rotor with angle $\theta$ and (scaled)
angular momentum $p$, no gravity — and kick it impulsively once per period
with a torque $K\sin\theta$. Between kicks $p$ is constant and $\theta$
advances by $p$; integrating across one period gives the *standard map*
{cite}`chirikov1979`

```{math}
:label: eq-nl-stdmap
p_{n+1} \;=\; p_n + K\sin\theta_n, \qquad
\theta_{n+1} \;=\; \theta_n + p_{n+1} \;\;(\mathrm{mod}\ 2\pi),
```

an area-preserving map of the cylinder that is *the* local model of any
Hamiltonian resonance. At $K = 0$ every orbit lies on a torus $p = \text{
const}$: [§2.10](hamilton-jacobi.ipynb)'s integrable picture, stroboscoped.
Small $K$ replaces the resonant torus $p = 0$ by a pendulum-like island of
half-width $2\sqrt K$ (Chirikov's resonance-width estimate), while
non-resonant tori survive as slightly wavy KAM curves that *bar vertical
transport*: an orbit cannot cross an invariant curve. As $K$ grows the
islands widen, KAM curves die one by one, and the most robust — the torus
with golden-mean winding number — dies last, at Greene's critical
$K_c = 0.971\,635\ldots$ {cite}`greene1979`. Beyond $K_c$ nothing bars the
way: $p$ performs a deterministic random walk, and for large $K$ the phases
$\theta_n$ decorrelate so effectively that
$\langle p_n^2 \rangle \approx 2 D n$ with the *quasilinear* coefficient
$D_{\rm ql} = K^2/4$ (the variance of one kick, $\langle K^2\sin^2\theta
\rangle = K^2/2$, accumulated as an uncorrelated walk). Correlations
between kicks decorate $D$ with oscillatory Bessel-function corrections
whose leading term is proportional to $J_2(K)$; we dodge the decoration by
measuring at a zero of $J_2$.

**Chaos in a flow: Lorenz.** Continuous-time chaos needs three dimensions,
and the canonical specimen is Lorenz's convection model {cite}`lorenz1963`

```{math}
:label: eq-nl-lorenz
\dot x = \sigma (y - x), \qquad
\dot y = x (\rho - z) - y, \qquad
\dot z = x y - \beta z,
```

at the classical parameters $\sigma = 10$, $\rho = 28$, $\beta = 8/3$.
Trajectories collapse onto the butterfly-shaped strange attractor and
separate on it at the rate given by the largest Lyapunov exponent,
$\lambda_1 \approx 0.9056$ (in inverse Lorenz time units): predictability
is lost with a half-life of order $\ln 2 / \lambda_1 \approx 0.77$ time
units, which is the quantitative content of the "butterfly effect". The
measurement tool is Benettin renormalization — run a reference and a
companion trajectory, and every interval $\tau$ record the logarithm of
their separation growth and rescale the companion back to distance $d_0$
along the current separation direction (rescaling before saturation is what
keeps the measurement in the linear regime the exponent describes.)

## Setup

Everything runs in dimensionless units: the maps on their natural
intervals, the standard map on the cylinder $\theta \in [0, 2\pi)$, Lorenz
in Lorenz's own time units. Randomness appears only in the choice of
ensemble initial conditions for the diffusion experiment and is seeded.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp
from scipy.special import jn_zeros

from ecp import animate, draw, validate

rng = np.random.default_rng(0)

FEIGENBAUM_DELTA = 4.669201609
FEIGENBAUM_ALPHA = -2.502907875
LORENZ_LAMBDA1 = 0.9056  # largest Lyapunov exponent at (10, 28, 8/3)


def logistic(x, r):
    """One step of the logistic map x -> r x (1 - x), eq-nl-logistic.

    The population toy model: growth at rate r checked by crowding (1 - x).
    Works elementwise on arrays, which is what lets a whole bifurcation
    diagram iterate as one vectorized sweep.

    Parameters
    ----------
    x : float or numpy.ndarray
        Current state(s) in [0, 1].
    r : float or numpy.ndarray
        Map parameter(s) in [0, 4].

    Returns
    -------
    float or numpy.ndarray
        The next state, same shape as the broadcast of ``x`` and ``r``.
    """
    return r * x * (1.0 - x)


def logistic_prime(x, r):
    """Derivative r (1 - 2x) of the logistic map with respect to x.

    The per-step error-amplification factor: |f'| < 1 contracts
    perturbations, |f'| > 1 stretches them, and its log-average is the
    Lyapunov exponent eq-nl-lyap.

    Parameters
    ----------
    x : float or numpy.ndarray
        State(s) at which to evaluate the derivative.
    r : float or numpy.ndarray
        Map parameter(s).

    Returns
    -------
    float or numpy.ndarray
        f'(x) = r (1 - 2x).
    """
    return r * (1.0 - 2.0 * x)


def attractor_period(r, burn=60000, look=64, tol=1e-7):
    """Period of the logistic attractor at parameter r, up to 32.

    Iterates past the transient, then compares the orbit to itself shifted
    by candidate periods 1, 2, 4, ..., 32; the first shift that reproduces
    the orbit to within ``tol`` is the period. Near a bifurcation the
    transient decays critically slowly, hence the long default burn-in.

    Parameters
    ----------
    r : float
        Map parameter.
    burn : int
        Transient iterations discarded before testing.
    look : int
        Length of the test orbit.
    tol : float
        Maximum |x_{n+p} - x_n| for period p to count as exact.

    Returns
    -------
    int
        The detected period (64 signals "longer than 32 or chaotic").
    """
    x = 0.31
    for _ in range(burn):
        x = logistic(x, r)
    orbit = []
    for _ in range(look):
        x = logistic(x, r)
        orbit.append(x)
    orbit = np.array(orbit)
    for p in (1, 2, 4, 8, 16, 32):
        if np.max(np.abs(orbit[p:] - orbit[:-p])) < tol:
            return p
    return 64


def superstable_params(f, df_dx, df_dr, n_max, guess1, x_c=0.5):
    """Superstable parameters R_0..R_n_max of a unimodal map, by Newton.

    Solves f^(2^n)(x_c; R) = x_c for R at each n. The composed map's
    r-derivative is propagated alongside the state by the tangent recursion
    (x, dx) -> (f(x), df_dr(x) + df_dx(x) * dx), which is exact forward
    sensitivity: no finite differencing of a 2^n-fold composition. Each
    R_n is seeded by geometric extrapolation with the Feigenbaum ratio, so
    Newton starts inside the correct basin (a poor seed converges to a
    *different* cascade's root — a real failure mode, not a hypothetical).

    Parameters
    ----------
    f : callable
        The map, ``f(x, r)``.
    df_dx : callable
        Partial derivative of the map with respect to x, ``df_dx(x, r)``.
    df_dr : callable
        Partial derivative of the map with respect to r, ``df_dr(x, r)``.
    n_max : int
        Deepest doubling level; returns n_max + 1 parameters.
    guess1 : float
        Starting guess for R_1 (R_0 is found from x_c's own superstable
        fixed-point condition; deeper levels self-seed).
    x_c : float
        The map's critical point (maximum), 1/2 for both maps used here.

    Returns
    -------
    numpy.ndarray
        The superstable parameters R_0 .. R_{n_max}.
    """
    Rs = []
    for n in range(n_max + 1):
        if len(Rs) >= 2:
            r = Rs[-1] + (Rs[-1] - Rs[-2]) / FEIGENBAUM_DELTA
        elif len(Rs) == 1:
            r = guess1
        else:
            r = 2.0 if f is logistic else 0.5
        for _ in range(80):
            x, dx = x_c, 0.0
            for _ in range(2**n):
                dx = df_dr(x, r) + df_dx(x, r) * dx
                x = f(x, r)
            step = (x - x_c) / dx
            r -= step
            if abs(step) < 1e-13:
                break
        Rs.append(r)
    return np.array(Rs)


def lyapunov_logistic(r_values, n_iter=20000, burn=1000):
    """Lyapunov exponent eq-nl-lyap of the logistic map, vectorized over r.

    Runs every parameter's orbit simultaneously as one NumPy array and
    accumulates the running mean of ln|f'(x)| after the burn-in. The
    absolute value's log diverges when an orbit lands exactly on x = 1/2;
    the clip guards the (measure-zero) event without biasing the average.

    Parameters
    ----------
    r_values : numpy.ndarray
        Parameters at which to estimate the exponent.
    n_iter : int
        Averaging iterations per parameter.
    burn : int
        Transient iterations discarded first.

    Returns
    -------
    numpy.ndarray
        Lyapunov exponents, one per entry of ``r_values``.
    """
    x = np.full_like(r_values, 0.31)
    for _ in range(burn):
        x = logistic(x, r_values)
    acc = np.zeros_like(r_values)
    for _ in range(n_iter):
        x = logistic(x, r_values)
        acc += np.log(np.clip(np.abs(logistic_prime(x, r_values)), 1e-300, None))
    return acc / n_iter


def standard_map_ensemble(K, n_steps, theta0, p0, record_every=0):
    """Iterate the standard map eq-nl-stdmap on an ensemble of orbits.

    Applies the kick p -> p + K sin(theta) then the free rotation
    theta -> theta + p (mod 2pi) to every ensemble member at once. Momentum
    is deliberately NOT wrapped: unbounded |p| growth is the signature of
    broken transport barriers, which is the physics being watched.

    Parameters
    ----------
    K : float
        Kick strength.
    n_steps : int
        Number of map iterations.
    theta0 : numpy.ndarray
        Initial angles of the ensemble.
    p0 : numpy.ndarray or float
        Initial momenta (broadcast against ``theta0``).
    record_every : int
        If positive, additionally return (steps, <p^2>) sampled at this
        stride, for diffusion fits.

    Returns
    -------
    tuple
        ``(theta, p, p_maxabs)`` after n_steps, where ``p_maxabs`` is each
        orbit's maximum |p| along the way; with ``record_every`` also
        ``(steps, mean_p2)``.
    """
    th = theta0.copy()
    p = np.broadcast_to(np.asarray(p0, dtype=float), th.shape).copy()
    p_maxabs = np.abs(p)
    steps, mean_p2 = [], []
    for n in range(1, n_steps + 1):
        p = p + K * np.sin(th)
        th = (th + p) % (2.0 * np.pi)
        p_maxabs = np.maximum(p_maxabs, np.abs(p))
        if record_every and n % record_every == 0:
            steps.append(n)
            mean_p2.append(float(np.mean(p**2)))
    if record_every:
        return th, p, p_maxabs, np.array(steps), np.array(mean_p2)
    return th, p, p_maxabs


def lorenz_rhs(t, s, sigma=10.0, rho=28.0, beta=8.0 / 3.0):
    """Right-hand side of the Lorenz system eq-nl-lorenz.

    Lorenz's three-mode truncation of convection: x the circulation speed,
    y and z temperature-field amplitudes, at the classical chaotic
    parameters by default.

    Parameters
    ----------
    t : float
        Time (unused; the system is autonomous — the signature matches
        ``scipy.integrate.solve_ivp``).
    s : array_like
        State ``(x, y, z)``.
    sigma, rho, beta : float
        The Prandtl-like, Rayleigh-like, and geometric parameters.

    Returns
    -------
    list of float
        The derivatives ``(dx/dt, dy/dt, dz/dt)``.
    """
    x, y, z = s
    return [sigma * (y - x), x * (rho - z) - y, x * y - beta * z]


def benettin_lambda1(s_start, tau=0.4, n_intervals=750, d0=1e-8):
    """Largest Lyapunov exponent of the Lorenz flow by Benettin renormalization.

    Integrates a reference trajectory and a companion offset by d0; every
    interval tau both advance (solve_ivp RK45, rtol 1e-9, atol 1e-11), the
    log of the separation growth is added to the running sum, and the
    companion is rescaled back to distance d0 along the current separation
    direction. The average log-growth per unit time converges to lambda_1.

    Parameters
    ----------
    s_start : numpy.ndarray
        Starting state, already on the attractor (transient removed).
    tau : float
        Renormalization interval; short enough that separation stays deep
        in the linear regime (d << attractor size ~ 40).
    n_intervals : int
        Number of intervals; total averaging time is tau * n_intervals.
    d0 : float
        Renormalized separation distance.

    Returns
    -------
    float
        The estimate of lambda_1 in inverse Lorenz time units.
    """
    s = s_start.copy()
    sp = s + np.array([d0, 0.0, 0.0])
    total = 0.0
    for _ in range(n_intervals):
        a = solve_ivp(lorenz_rhs, (0, tau), s, rtol=1e-9, atol=1e-11).y[:, -1]
        b = solve_ivp(lorenz_rhs, (0, tau), sp, rtol=1e-9, atol=1e-11).y[:, -1]
        d = float(np.linalg.norm(b - a))
        total += np.log(d / d0)
        sp = a + (b - a) * (d0 / d)
        s = a
    return total / (n_intervals * tau)

## Exercise 1 — The logistic laboratory

Everything begins with {eq}`eq-nl-logistic` and its fixed points. Setting
$x^* = r x^* (1 - x^*)$ gives $x^* = 1 - 1/r$, with derivative
$f'(x^*) = 2 - r$; the linearization argument of the theory section says
this fixed point holds the population steady only while $|2 - r| < 1$.
Past $r = 3$ the map settles instead onto the period-2 cycle whose two
points are the roots of $f(f(x)) = x$ that are *not* fixed points of $f$,
which factor out to

$$x_\pm \;=\; \frac{(r + 1) \pm \sqrt{(r - 3)(r + 1)}}{2r},$$

real exactly when $r \ge 3$: the cycle is *born* at the bifurcation.

**Part a)** At $r = 2.9$, iterate {eq}`eq-nl-logistic` from $x_0 = 0.2$
for $500$ steps and verify the orbit converges to $x^* = 1 - 1/2.9$ to
$|x_{500} - x^*| < 10^{-10}$, and that the measured convergence is
governed by $f'(x^*) = 2 - r = -0.9$ (compare `logistic_prime(x*, 2.9)`
to $-0.9$ at `rtol=1e-12`). At $r = 3.2$, verify the same fixed point now
*repels* ($|f'(x^*)| > 1$) and that $2000$ iterations from $x_0 = 0.2$
land on the two-cycle: the final two iterates match $x_\pm$ above
(`numpy.sort` then `validate.close`, `rtol=1e-10`).

**Part b)** Draw the cobweb construction at $r = 2.8$ (spiral into the
fixed point), $r = 3.4$ (the period-2 square), and $r = 3.9$ (chaotic
tangle): plot the parabola $f(x)$ and the diagonal $y = x$, then the
staircase segments $(x_n, x_n) \to (x_n, x_{n+1}) \to (x_{n+1}, x_{n+1})$
for $60$ iterations from $x_0 = 0.2$.

**Part c)** Build the orbit (bifurcation) diagram: for each of $1400$
parameters $r \in [2.8, 4.0]$, iterate a vectorized ensemble $800$ burn-in
steps and scatter the next $300$ iterates. Then *measure* the first two
bifurcation points by bisection on `attractor_period` (period $1 \to 2$
between $r = 2.8$ and $3.2$; period $2 \to 4$ between $3.3$ and $3.5$) and
verify them against the exact $r_1 = 3$ and $r_2 = 1 + \sqrt 6$
(`rtol=1e-3`; the residual is the period-detector's finite tolerance
fighting critical slowing-down, not the mathematics).

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    conv_err < 1e-10,
    "at r = 2.9 the orbit converges onto x* = 1 - 1/r",
    f"|x_500 - x*| = {conv_err:.1e}",
)
validate.close(
    slope_a,
    -0.9,
    "and the measured stability factor is f'(x*) = 2 - r = -0.9 exactly",
    rtol=1e-12,
)
validate.check(
    abs(logistic_prime(xstar_b, r_b)) > 1.0,
    "at r = 3.2 the same fixed point has |f'| > 1: it repels",
    f"|f'| = {abs(logistic_prime(xstar_b, r_b)):.3f}",
)
validate.close(
    two_cycle_num,
    two_cycle_exact,
    "and the orbit lands on the period-2 cycle at its closed-form points "
    "x± = ((r+1) ± √((r-3)(r+1)))/(2r)",
    rtol=1e-10,
)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    r1_meas,
    3.0,
    "the first period-doubling, located by bisection on the attractor "
    "period, sits at r1 = 3",
    rtol=1e-3,
)
validate.close(
    r2_meas,
    1.0 + np.sqrt(6.0),
    "and the second at r2 = 1 + √6 = 3.44949",
    rtol=1e-3,
)

## Exercise 2 — Feigenbaum's constants, measured

The doubling cascade of {numref}`fig-nl-orbit-diagram` accumulates
geometrically, and {eq}`eq-nl-delta` defines the rate. To measure it we
need the superstable parameters $R_n$ to many digits, and the right tool
is Newton's method on the defining condition $f^{(2^n)}(x_c; R) = x_c$
with $x_c = 1/2$. Newton needs the derivative of a $2^n$-fold composition
with respect to $R$, and the clean way to get it is the *tangent
recursion*: propagate the pair $(x, \partial x / \partial R)$ through the
map together,

$$x \mapsto f(x; R), \qquad
\frac{\partial x}{\partial R} \mapsto
\frac{\partial f}{\partial R}(x; R)
+ \frac{\partial f}{\partial x}(x; R)\,\frac{\partial x}{\partial R},$$

which is exact forward sensitivity (the same chain-rule idea that
propagates error bars, and a baby version of what automatic
differentiation does). Seeding matters: each $R_n$ is predicted from the
previous two by the Feigenbaum ratio itself, because a careless seed lets
Newton converge onto a *different* root of $f^{(2^n)}(x_c) = x_c$ (every
shallower superstable parameter is one) and silently wreck the ladder.

**Part a)** Implement the tangent-recursion Newton solver and compute
$R_0, \ldots, R_8$ for the logistic map. Two rungs are known in closed
form and certify the solver: $R_0 = 2$ (where $f(1/2) = 1/2$ exactly) and
$R_1 = 1 + \sqrt 5$ (where the 2-cycle contains $1/2$); verify both to
`atol=1e-9`. **Write this one yourself** — the implementation is the
lesson.

**Part b)** Form the ratios $\delta_n$ of {eq}`eq-nl-delta` and verify the
deepest one, $\delta_7$, agrees with $\delta = 4.669\,201\,609$ to
`rtol=1e-4`. Estimate $R_\infty$ by geometric extrapolation
$R_\infty \approx R_8 + (R_8 - R_7)/(\delta - 1)$ and plot
$R_\infty - R_n$ on a log axis: the straight line *is* the geometric
convergence.

**Part c)** Measure $\alpha$: at each $R_n$ the cycle point nearest the
maximum sits a distance $d_n = f^{(2^{n-1})}(x_c; R_n) - x_c$ away, and
successive ratios $d_n / d_{n+1}$ converge to $\alpha = -2.502\,908$
(note the sign: the nearest return alternates sides). Verify the deepest
ratio to `rtol=1e-3`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    R_log[0],
    2.0,
    "the solver certifies on the closed-form rung R_0 = 2",
    rtol=0.0,
    atol=1e-9,
)
validate.close(
    R_log[1],
    1.0 + np.sqrt(5.0),
    "and on R_1 = 1 + √5, the superstable 2-cycle",
    rtol=0.0,
    atol=1e-9,
)
validate.close(
    delta_n[-1],
    FEIGENBAUM_DELTA,
    "eight doublings deep, the ratio of successive gaps is Feigenbaum's "
    "delta = 4.669201609",
    rtol=1e-4,
)
validate.close(
    alpha_n[-1],
    FEIGENBAUM_ALPHA,
    "and the nearest-return distances shrink by alpha = -2.502908 per "
    "doubling, sign and all",
    rtol=1e-3,
)

## Exercise 3 — Universality: a different map, the same constant

The measurement so far could be a curiosity of the parabola. Feigenbaum's
claim {cite}`feigenbaum1978` is far stronger: *every* one-hump map with a
quadratic maximum doubles its way to chaos with the same $\delta$ and
$\alpha$, because near the accumulation point the dynamics is controlled
by a renormalization fixed point that has forgotten the original map
entirely (the same logic — details washed out under repeated
coarse-graining — will return as the explanation of universal critical
exponents in [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb)).
The honest test is to measure again in a map that shares nothing with the
logistic map but the shape of its top: the sine map

$$x_{n+1} = r\,\sin(\pi x_n), \qquad x \in [0, 1],\;
r \in [0, 1],$$

transcendental where the logistic map is polynomial, with its cascade
living on a completely different parameter interval.

**Part a)** Reuse `superstable_params` with $f = r\sin(\pi x)$,
$\partial f/\partial x = r\pi\cos(\pi x)$, $\partial f/\partial r =
\sin(\pi x)$, seeding $R_1$ at $0.78$, to compute the sine map's
$R_0, \ldots, R_8$. Verify $R_0 = 1/2$ exactly (where $r\sin(\pi/2) =
1/2$) to `atol=1e-10`, and verify the cascade is *not* the logistic one:
its $R_1$ differs from the logistic $R_1 = 1 + \sqrt 5$ by more than $2$.

**Part b)** Form the sine map's $\delta_n$ ratios and verify $\delta_7$
agrees with the logistic measurement of Exercise 2 to `rtol=1e-3`: two
maps, two cascades, one constant. Plot the two orbit diagrams one above
the other over their own parameter ranges; the point of the figure is
that they are the same picture stretched differently.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    R_sin[0],
    0.5,
    "the sine map's R_0 = 1/2 exactly (its superstable fixed point)",
    rtol=0.0,
    atol=1e-10,
)
validate.check(
    abs(R_sin[1] - R_log[1]) > 2.0,
    "the two cascades live on different parameter intervals: the maps "
    "share nothing but the shape of their maximum",
    f"|R_1(sine) - R_1(logistic)| = {abs(R_sin[1] - R_log[1]):.3f}",
)
validate.close(
    delta_sin[-1],
    delta_n[-1],
    "yet their doubling ratios agree: Feigenbaum's constant belongs to the "
    "transition, not the equation",
    rtol=1e-3,
)

## Exercise 4 — The Lyapunov exponent across the diagram

The orbit diagram shows *where* the attractor is; the Lyapunov exponent
{eq}`eq-nl-lyap` says how *predictable* it is, and laying the two on the
same parameter axis turns the bifurcation diagram into an instrument
panel. The conjugacy benchmark makes one point of the curve exact: with
$x = \sin^2(\pi y/2)$, the $r = 4$ logistic map becomes the tent map on
$y$, which stretches every interval by the factor $2$, so
$\lambda(4) = \ln 2$ precisely.

**Part a)** Compute $\lambda(r)$ with `lyapunov_logistic` on the $1400$
parameters of Exercise 1's grid ($20\,000$ averaging iterations after a
$1000$-step transient), and plot it beneath the orbit diagram with the
shared $r$ axis and a zero line.

**Part b)** Verify the three signatures: (i) the exact benchmark
$\lambda(4) = \ln 2$ at `rtol=1e-3` (with $2\times10^5$ iterations for
the single-point average); (ii) $\lambda = 0$ at the bifurcation $r = 3$
to `atol=2e-3` (the handover moment: the cycle is marginally stable);
(iii) deep in the period-3 window, $\lambda(3.835) < -0.25$ — order
*inside* chaos, exactly where {numref}`fig-nl-orbit-diagram` shows the
clear vertical band.

**Part c)** Measure how much of the chaotic regime is actually chaotic:
the fraction of the $400$-point grid on $[3.57, 4.0]$ with $\lambda > 0$,
which should land between $0.85$ and $0.97$ — the windows puncture the
chaos everywhere ({numref}`fig-nl-lyap` shows them as downward spikes),
but they puncture it on a small fraction of parameters.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    lam4,
    np.log(2.0),
    "the exact benchmark: at r = 4 the tent-map conjugacy fixes " "lambda = ln 2",
    rtol=1e-3,
)
validate.check(
    abs(lam3) < 2e-3,
    "at the r = 3 bifurcation the exponent is zero: marginal stability at "
    "the handover",
    f"lambda(3) = {lam3:+.1e}",
)
validate.check(
    lam_win < -0.25,
    "deep in the period-3 window the exponent is decisively negative: "
    "order inside chaos",
    f"lambda(3.835) = {lam_win:+.3f}",
)
validate.check(
    0.85 < frac_chaotic < 0.97,
    "chaos beyond r_inf is punctured by periodic windows on a small but "
    "non-zero fraction of parameters",
    f"fraction = {frac_chaotic:.3f}",
)

## Exercise 5 — The kicked rotor: tori die one by one

Now the Hamiltonian act. The kicked rotor is the cleanest bridge from
[§2.10](hamilton-jacobi.ipynb): a free rotor *is* an action-angle system
(its momentum is the action, its angle advances uniformly), and the
periodic kick is the perturbation that decides which tori survive.
{numref}`fig-nl-rotor` shows the setup; stroboscoping at the kick period
gives the standard map {eq}`eq-nl-stdmap`, and everything KAM theory has
to say about weakly perturbed integrable systems plays out on its
cylinder.

In [ ]:
# (solution hidden on the public site)


Three experiments, three regimes.

**Part a) The near-integrable limit and the resonance width.** At
$K = 0.05$, iterate an ensemble of $100$ orbits started on the resonant
torus ($p_0 = 0$, angles `numpy.linspace(0.1, 2π-0.1, 100)`) for $5000$
steps of {eq}`eq-nl-stdmap` with `standard_map_ensemble`, and record each
orbit's maximum $|p|$ excursion. Chirikov's pendulum approximation says the resonance at
$p = 0$ becomes an island of half-width exactly $2\sqrt K$; verify the
ensemble's largest excursion lands between $0.8$ and $1.05$ times
$2\sqrt K = 0.447$: the orbits fill the island right up to its
separatrix and are stopped there by the surviving KAM curves.

**Part b) The last torus.** Iterate the same ensemble with
`standard_map_ensemble` at $K = 0.9$ (just below Greene's $K_c = 0.9716$)
and at $K = 1.2$ (just above) for $5000$ steps, reading off each run's
maximum $|p|$. Verify the confinement verdict: at $K = 0.9$ the maximum $|p|$
over the whole ensemble stays below $2\pi$ (some KAM curve, the
golden-mean torus among the last, still bars the first cylinder period),
while at $K = 1.2$ it exceeds $2\pi$ — the barrier is gone and transport
is global. Plot the four phase portraits $K = 0.5, 0.9, 1.2, 2.0$ (a
$24\times24$ grid of initial conditions, $400$ strobe points each,
$p$ wrapped to $[-\pi,\pi]$ for display) to watch the sea swallow the
curves.

**Part c) Deterministic diffusion.** Far above $K_c$ the momentum walk
should be diffusive with $D \approx K^2/4$. The Bessel-function
correlation corrections vanish at zeros of $J_2$, so measure exactly
there: at $K = 8.4172$ (the second zero, from
`scipy.special.jn_zeros(2, 2)`), evolve $20\,000$ orbits from uniformly
random seeded angles ($p_0 = 0$) for $2000$ steps, recording
$\langle p^2 \rangle$ every $100$. Verify (i) growth is diffusive: the
`numpy.polyfit` slope of $\ln\langle p^2\rangle$ against $\ln n$ is $1$
within $\pm 0.05$, and (ii) the measured $D = \langle p^2\rangle / (2n)$
at $n = 2000$ matches the quasilinear $K^2/4$ to within $15\%$: a
deterministic map, diffusing like a random walk, at the rate a
random-phase argument predicts.

```{admonition} With your assistant
:class: tip
The four-panel phase-portrait grid of Part b) is honest boilerplate:
describe the layout to your assistant (a 24×24 grid of initial
conditions on the cylinder, 400 strobe points each, momentum wrapped to
$[-\pi,\pi]$, one panel per $K$) and let it write the plotting loop. The
check is yours, and it is physical: rerun Part a) at $K = 0.02$ and
confirm the island's measured half-width still tracks $2\sqrt{K} =
0.283$. The scaling law, not the plot, is the physics.
```

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    0.8 < width_ratio < 1.05,
    "the K = 0.05 island fills to its separatrix: maximum excursion "
    "matches Chirikov's half-width 2√K",
    f"ratio = {width_ratio:.3f}",
)
validate.check(
    float(pmax_09.max()) < 2.0 * np.pi < float(pmax_12.max()),
    "the last KAM torus holds at K = 0.9 and is gone at K = 1.2, "
    "bracketing Greene's K_c = 0.9716",
    f"max|p|: {float(pmax_09.max()):.2f} vs {float(pmax_12.max()):.2f}, "
    f"barrier 2π = {2 * np.pi:.2f}",
)
validate.check(
    abs(slope_diff - 1.0) < 0.05,
    "above K_c the momentum spread grows linearly in n: deterministic "
    "motion, diffusive bookkeeping",
    f"slope = {slope_diff:.3f}",
)
validate.close(
    D_meas,
    D_ql,
    "and at a J_2 zero the diffusion coefficient is quasilinear K²/4, "
    "as the random-phase argument predicts",
    rtol=0.15,
)

## Exercise 6 — Lorenz: measuring the butterfly

The final act moves from maps to a flow. The Lorenz system
{eq}`eq-nl-lorenz` at $(\sigma, \rho, \beta) = (10, 28, 8/3)$ is
dissipative (volumes contract at the constant rate $\sigma + 1 + \beta =
41/3$ per time unit), so trajectories fall onto an attractor; the
surprise of 1963 is that the attractor is neither a point nor a loop but
a *strange* set on which motion separates exponentially forever.

**Part a) The attractor and the butterfly effect.** Integrate
{eq}`eq-nl-lorenz` with `scipy.integrate.solve_ivp` (RK45,
`rtol=1e-12`, `atol=1e-14`) from $(1, 1, 20)$: discard $25$ time units of
transient, then follow the attractor for $25$ more from the landing state
$\mathbf s_0$, sampling $2500$ points, and plot the $x$–$z$ butterfly.
Repeat from $\mathbf s_0 + (10^{-9}, 0, 0)$ and plot
$\ln\lVert\Delta\mathbf s(t)\rVert$: the separation must grow *linearly
in the log* until it saturates at the attractor's size. Fit the growth
rate over the window from $t = 2$ until the separation first exceeds
$1$ (`numpy.polyfit`, degree 1).

**Part b) The exponent, twice.** Measure $\lambda_1$ properly with
`benettin_lambda1` ($\tau = 0.4$, $750$ renormalization intervals: $300$
time units on the attractor) and verify it against the literature value
$0.9056$ to `rtol=5e-2`; then verify the *cross-method* consistency: the
Part a) divergence-fit rate agrees with the Benettin value to
`atol=0.15`. Two measurements of one number, one from a single
separation event, one from three hundred renormalized ones.

**Part c) Lorenz's own argument.** Record every successive local maximum
of $z(t)$ over $120$ time units and plot each maximum against the next:
the three-dimensional flow collapses onto a *one-dimensional* tent-shaped
return map (this is Lorenz's 1963 construction). Estimate the local slope
magnitudes with `numpy.diff` on the sorted pairs and verify that at least
$98\%$ exceed $1$: an everywhere-expanding return map can hold no stable
periodic orbit, which is the cleanest one-line proof that the attractor's
irregularity is structural, not transient.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    lam1_benettin,
    LORENZ_LAMBDA1,
    "Benettin renormalization over 300 time units reproduces the Lorenz "
    "system's largest Lyapunov exponent 0.9056",
    rtol=5e-2,
)
validate.check(
    abs(rate_fit - lam1_benettin) < 0.15,
    "and the single-shot divergence fit agrees with the renormalized "
    "measurement: two methods, one exponent",
    f"fit {rate_fit:.3f} vs Benettin {lam1_benettin:.3f}",
)
validate.check(
    len(z_max) > 100 and frac_expanding >= 0.98,
    "Lorenz's return map of successive z-maxima is expanding "
    "(|slope| > 1) essentially everywhere: no stable periodic orbit can "
    "exist on the attractor",
    f"{len(z_max)} maxima, expanding fraction {frac_expanding:.3f}",
)

## Notebook summary

- The logistic map's fixed point $x^* = 1 - 1/r$ obeyed its linearization
  exactly ($f'(x^*) = 2 - r$, convergence at $r = 2.9$, repulsion and the
  closed-form 2-cycle at $r = 3.2$), and bisection on the attractor
  period located the first two doublings at $r_1 = 3$ and
  $r_2 = 1 + \sqrt 6$ to a part in a thousand.
- Newton's method with exact tangent-recursion sensitivities pinned the
  superstable cascade $R_0 \ldots R_8$ (certified on $R_0 = 2$ and
  $R_1 = 1 + \sqrt 5$), delivering $\delta_7 = 4.66919$ against
  Feigenbaum's $4.669\,201\,609$ and $\alpha = -2.5029$; the sine map's
  independent cascade gave $\delta_7 = 4.66915$ — the same constant from
  an unrelated formula, which is what "universal" means.
- The Lyapunov exponent $\lambda(r) = \langle\ln|f'|\rangle$ hit its
  exact benchmark $\lambda(4) = \ln 2$, vanished at the $r = 3$
  bifurcation, plunged negative inside the period-3 window, and was
  positive on $91\%$ of the post-accumulation parameter range.
- The standard map delivered KAM theory's whole arc: the $p = 0$ island
  filled to Chirikov's half-width $2\sqrt K$ at $K = 0.05$; the ensemble
  stayed confined below $2\pi$ at $K = 0.9$ and broke through at
  $K = 1.2$, bracketing Greene's $K_c = 0.9716$; and at $K = 8.417$ (a
  $J_2$ zero) the deterministic momentum walk diffused with slope
  $0.996$ and $D = K^2/4$ to under two percent.
- The Lorenz system's largest Lyapunov exponent came out as $0.908$ by
  Benettin renormalization and $0.898$ by a single divergence fit,
  against the literature $0.9056$; and successive $z$-maxima collapsed
  onto Lorenz's one-dimensional return map, expanding everywhere — the
  attractor holds no stable orbit, by construction rather than by
  assertion.

## Outlook

- **Universality beyond maps.** The Feigenbaum constants measured here
  govern the period-doubling cascade of the driven pendulum of
  [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb), of
  Rayleigh–Bénard convection cells, and of driven nonlinear circuits;
  Libchaber's helium experiments measured $\delta$ within a few percent
  of the value the Newton ladder of Exercise 2 produced. The renormalization argument
  behind it is the same coarse-graining logic that makes critical
  exponents universal in
  [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb).
- **The measure of chaos.** Greene's residue method {cite}`greene1979`
  pins $K_c$ to six digits by watching periodic orbits whose winding
  numbers are continued-fraction approximants of the golden mean; the
  golden torus dies last because its winding number is the *hardest* to
  approximate by rationals. KAM theory turns that arithmetic into a
  theorem.
- **Chaos with a deadline.** [§1.8](../01-elementary-mechanics/solar-system.ipynb)
  promised the tools to quantify the solar system's sensitivity: Laskar's
  integrations give the inner planets a Lyapunov time near $5$ Myr — the
  solar system is a Lorenz butterfly with a five-million-year wingbeat,
  predictable in the same conditional sense as the weather.
- **From chaos to statistics.** [§5.5](../05-classical-stat-mech/ergodicity.ipynb)
  picks up exactly where the broken tori leave off: once no invariant
  curve confines a trajectory, time averages can equal ensemble averages,
  and mechanics hands off to statistical mechanics. That handoff is the
  course's next volume in miniature.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()